# Sesión 7 — Buenas prácticas y validación: no le creas a la IA sin chequear

Mirá, llegamos a la sesión que más me importa de todo el curso. Y no es por capricho.

En las sesiones anteriores le pedimos código a la IA y, casi siempre, "funcionó": corrió, no tiró error, nos devolvió un número, un gráfico, una tabla. Y ahí está la TRAMPA. Que un código corra NO significa que esté bien. Significa apenas que no se rompió. Son dos cosas completamente distintas.

Cuando vos investigás un conflicto portuario, una huelga, la prensa de una época, cada número que reportás es una afirmación sobre el pasado. Si decís "la palabra *paro* aparece 30 veces en la prensa", eso es un dato que alguien va a citar, que va a entrar en tu argumento. Si ese 30 está mal porque el código contó de más, tu argumento se construye sobre arena.

La IA se equivoca. Alucina funciones que no existen. Escribe código plausible que hace algo SUTILMENTE distinto de lo que pediste. Y lo hace con seguridad absoluta, sin avisarte. El rigor no lo pone la herramienta: lo ponés vos, que sos quien investiga. Hoy aprendemos a desconfiar con método.

> Trabajamos sobre `../datos/notas_prensa.csv`, un corpus SINTÉTICO (ficticio) de 22 notas que narran un conflicto en plantas pesqueras: tensión, asamblea, paro, negociación y acuerdo. Es chico A PROPÓSITO: con 22 notas todavía podés validar a mano. Esa es justamente la idea de hoy.

## Primero, cargamos el corpus y lo miramos

Antes de validar nada, tenemos que poder VER los datos. Validar a ciegas no es validar.

In [ ]:
# Usamos pandas para manejar la tabla de notas
import pandas as pd

# Leemos el corpus de notas de prensa (ruta relativa desde esta carpeta)
notas = pd.read_csv("../datos/notas_prensa.csv")

# Cuántas notas tenemos y qué columnas
print(notas.shape)
print(list(notas.columns))

In [ ]:
# Miramos las primeras filas para entender con qué trabajamos
notas[["id", "fecha", "titular"]].head(5)

22 notas, columnas claras. Bien. Tenemos el terreno bajo control.

## El prompt: el pedido que le hicimos a la IA

Imaginá que queremos saber cuántas veces aparece la palabra **paro** en los textos del corpus, para medir la intensidad del conflicto a lo largo del tiempo. Le escribimos a la IA algo como esto:

> "Tengo un DataFrame de pandas llamado `notas` con una columna `texto`. Contá cuántas veces aparece la palabra *paro* en todos los textos y devolveme el total."

## El código "que la IA nos devolvió"

Y la IA, servicial, nos devolvió esto. Léelo despacio, como si fueras a corregir el trabajo de otra persona:

In [ ]:
# --- VERSIÓN DE LA IA (¡ATENCIÓN: tiene un error sutil!) ---

# Cuenta cuántas veces aparece "paro" en cada texto y suma el total
total_paro_ia = notas["texto"].str.count("paro").sum()

print(total_paro_ia)

Nos da un número. Corre sin error. Se ve perfecto. Y si lo copiás y lo pegás en tu artículo, estás reportando un dato MAL.

## Cómo detectamos el problema (esto es lo que hay que aprender)

¿Cómo nos damos cuenta? No por intuición mágica, sino con TRES movimientos concretos que podés repetir siempre.

### Movimiento 1: leer el código crítico, no solo correrlo

La línea clave es `notas["texto"].str.count("paro")`. La pregunta honesta es: ¿qué busca exactamente `.str.count`? Busca el PATRÓN `"paro"` como subcadena, es decir, esa secuencia de letras AUNQUE esté metida adentro de otra palabra.

Pensá qué palabras de un texto sobre conflictos contienen la cadena `paro`: **partici*paro*n** (participaron), **am*paro*** (amparo), **dis*paro*** (disparo), **re*paro*** (reparo)... Todas esas suman al conteo sin ser la palabra *paro*. En NUESTRO corpus, la que se cuela es **participaron**.

### Movimiento 2: validar a mano un caso sobre el corpus chico

Acá está la ventaja de un corpus de 22 notas: podemos chequear de verdad. Vamos a buscar QUÉ textos contienen la cadena "paro" y mirarlos con nuestros ojos.

In [ ]:
# Traemos las filas donde aparece la cadena "paro" y vemos el texto completo
sospechosos = notas[notas["texto"].str.contains("paro")][["id", "texto"]]
sospechosos

In [ ]:
# Para el ojo humano: extraemos TODOS los fragmentos que la máquina contó como "paro"
import re

# Unimos todos los textos y buscamos cualquier palabra que contenga la cadena "paro"
todo_el_texto = " ".join(notas["texto"].tolist())
matches = re.findall(r"\w*paro\w*", todo_el_texto)

# Las mostramos sin repetir, para verlas con nuestros ojos
sorted(set(matches))

¿Ves? Entre los matches aparece **participaron**, que NO es la palabra *paro* del conflicto sindical: es el verbo *participar* en pasado. La máquina la contó igual. El número estaba inflado. (En otros corpus se te van a colar *amparo*, *disparo*, *reparo*: el problema es siempre el mismo, la búsqueda por subcadena.)

### Movimiento 3: comparar conteos

Ahora contemos bien y comparemos. La diferencia es la prueba del delito.

## La versión corregida: buscar la PALABRA completa

El arreglo es decirle a la máquina "quiero la palabra *paro*, no la cadena de letras". En expresiones regulares, los límites de palabra se escriben `\b`.

In [ ]:
# --- VERSIÓN CORREGIDA ---

# \b marca un "límite de palabra": antes y después de "paro" no puede haber letra.
# Además normalizamos a minúsculas por si hubiera "Paro" al inicio de una oración.
patron = r"\bparo\b"

total_paro_ok = notas["texto"].str.lower().str.count(patron).sum()

print(total_paro_ok)

In [ ]:
# La prueba del delito: cuántas coincidencias de más contaba la versión de la IA
pd.DataFrame({
    "version_ia": [total_paro_ia],
    "version_correcta": [total_paro_ok],
    "diferencia": [total_paro_ia - total_paro_ok]
})

Esa `diferencia` son las palabras que la versión ingenua contó de más. Si fuera cero, igual el chequeo valió la pena: ahora SABÉS que está bien, no lo SUPONÉS.

## Segundo peligro: la función que no existe (alucinación)

Hay un error todavía más traicionero: la IA inventa funciones que SUENAN perfectas pero no existen. A esto le decimos *alucinación*. Por ejemplo, podría decirte con total seguridad: "usá `notas[\"texto\"].str.count_words(\"paro\")` para contar palabras completas". Suena lógico, ¿no? El problema es que **ese método no existe** en pandas.

¿Cómo se valida esto en dos segundos? Preguntándole a Python, no a la IA:

In [ ]:
# ¿Existe el método que la IA me sugirió? Le preguntamos directamente a Python.
# hasattr nos dice si el accesor .str de pandas tiene de verdad ese método.
print("¿Existe .str.count_words?", hasattr(notas["texto"].str, "count_words"))
print("¿Existe .str.count?      ", hasattr(notas["texto"].str, "count"))

El primero devuelve `False`: no existe. El segundo, `True`: ese sí. La regla es simple: **antes de confiar en una función, verificá que exista de verdad**. En un notebook podés usar `hasattr(...)`, escribir `help(notas["texto"].str.count)`, o tocar TAB después del punto para ver la lista real de métodos. Si la IA te nombra una función, esa es una afirmación que tenés que comprobar, no creer.

## Cómo documentar para que esto sea reproducible

Encontraste el error, lo corregiste, validaste. Falta el último paso, el que diferencia un cálculo casero de un resultado de investigación: dejar constancia de QUÉ hiciste, con QUÉ datos y CÓMO se vuelve a correr. Si dentro de seis meses (o un colega, o un evaluador) no puede reproducir tu número, el número no vale.

Una buena nota de reproducibilidad responde tres preguntas: qué datos, qué versiones, cómo correr.

In [ ]:
# --- NOTA DE REPRODUCIBILIDAD ---
# Qué hace este script: cuenta apariciones de la PALABRA "paro" (no subcadenas)
#   en la columna 'texto' del corpus de prensa, normalizando a minúsculas.
# Datos: ../datos/notas_prensa.csv (corpus SINTÉTICO, 22 notas, conflicto pesquero).
# Cómo correr: abrir este notebook y ejecutar las celdas en orden, de arriba
#   hacia abajo. Requiere: Python 3 y pandas.

# Dejamos registro de las versiones con las que ESTE resultado fue calculado.
# Si alguien obtiene otro número, lo primero a revisar son estas versiones.
import sys
import platform

print("Python:", sys.version.split()[0])
print("Sistema:", platform.platform())
print("pandas:", pd.__version__)

Esa lista de versiones es oro: deja por escrito tu Python y tus paquetes. Pegala en un apéndice, en un README, en una nota al pie. Es barato y te salva.

## Que hace y cómo lo validamos (resumen del método)

Repasemos el método, porque es lo que te llevás de hoy:

1. **Leé el código crítico**, no solo lo corras. La línea que produce tu número es la que más atención merece.
2. **Validá a mano sobre el corpus chico**: buscá los casos, miralos con tus ojos, comprobá un ejemplo concreto.
3. **Compará conteos**: la versión ingenua contra la corregida. La diferencia te muestra el error.
4. **Verificá que las funciones existan**: `hasattr()`, `help()`, la documentación. No creas en funciones por cómo suenan.
5. **Documentá**: qué datos, qué versiones, cómo correr. Reproducibilidad o no es investigación.

## Para jugar

Probá pedirle a la IA estas variaciones y validá cada resultado con el método de arriba:

1. "Contá ahora la palabra *huelga* y la palabra *acuerdo* por separado, usando límites de palabra, y mostrame las dos al lado." (Fijate si vuelve a usar `\b` o si recae en la búsqueda por subcadena.)
2. "Revisá este código y decime qué puede salir mal antes de que lo use." Pegale la versión ingenua del principio y mirá si DETECTA el problema de *participaron*. A veces lo encuentra; a veces no. Por eso el chequeo final siempre es tuyo.